# Feature-Group Ablation: Temporal Drift Resistance

Answers **RQ3**: *Which behavioral representation is most temporally stable?*

Design:
- **Leave-one-group-out**: remove one group, retrain XGBoost on KronoDroid, test on AndroZoo 2024–2026
- **Single-group-only**: train with only that group
- Full model = baseline (0.701 macro-F1)

Feature groups (from Table II):
- Raw Counters     : indices 0–78   (79 features)
- Derived Features : indices 79–114 (36 features)
- Temporal Features: indices 115–136 (22 features)
- Sequence Features: indices 137–164 (28 features)

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

BASE = Path('/home/tan/GitHub/paper-kangal-dynamic/dynamic_ieee_paper')
KRONO_DIR  = BASE / 'krono_dataset'
AZOO_DIR   = BASE / 'androzoo_dataset'
RESULT_DIR = BASE / 'deneyler/results/ablation'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

META_COLS = ['package_name', 'label', 'timestamp']

In [2]:
# Load KronoDroid
df_krono = pd.concat([
    pd.read_csv(KRONO_DIR / 'krono_malware.csv'),
    pd.read_csv(KRONO_DIR / 'krono_benign.csv', engine='python')
], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Load AndroZoo (3000 malware + 3000 benign, balanced)
df_azoo = pd.concat([
    pd.read_csv(AZOO_DIR / 'androzoo_malware.csv').sample(n=3000, random_state=SEED),
    pd.read_csv(AZOO_DIR / 'androzoo_benign.csv',  engine='python').sample(n=3000, random_state=SEED)
], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

feature_cols = [c for c in df_krono.columns if c not in META_COLS]
assert len(feature_cols) == 165, f'Expected 165 features, got {len(feature_cols)}'

y_krono = (df_krono['label'] == 'malware').astype(int).to_numpy()
y_azoo  = (df_azoo['label']  == 'malware').astype(int).to_numpy()

X_krono = df_krono[feature_cols].fillna(df_krono[feature_cols].median())
X_azoo  = df_azoo[feature_cols].fillna(X_krono.median())  # fit imputer on KronoDroid

# 85/15 KronoDroid train/val split (same as paper)
X_train, X_val, y_train, y_val = train_test_split(
    X_krono, y_krono, test_size=0.15, stratify=y_krono, random_state=SEED
)

print(f'KronoDroid train: {len(X_train):,}  val: {len(X_val):,}')
print(f'AndroZoo test   : {len(X_azoo):,}')

KronoDroid train: 12,195  val: 2,153
AndroZoo test   : 6,000


In [3]:
# Feature group definitions
GROUPS = {
    'Raw Counters'     : feature_cols[0:79],
    'Derived Features' : feature_cols[79:115],
    'Temporal Features': feature_cols[115:137],
    'Sequence Features': feature_cols[137:165],
}
for g, cols in GROUPS.items():
    print(f'{g}: {len(cols)} features')

Raw Counters: 79 features
Derived Features: 36 features
Temporal Features: 22 features
Sequence Features: 28 features


In [4]:
def run_xgb(X_tr, y_tr, X_te, y_te, X_az, y_az):
    model = XGBClassifier(
        n_estimators=600, max_depth=5, learning_rate=0.04,
        use_label_encoder=False, eval_metric='logloss',
        random_state=SEED, n_jobs=-1, verbosity=0
    )
    model.fit(X_tr, y_tr)
    # KronoDroid val
    y_pred_val = model.predict(X_te)
    krono_f1   = f1_score(y_te, y_pred_val, average='macro')
    # AndroZoo temporal
    y_pred_az  = model.predict(X_az)
    az_f1      = f1_score(y_az, y_pred_az, average='macro')
    tn, fp, fn, tp = confusion_matrix(y_az, y_pred_az).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    delta = az_f1 - krono_f1
    return krono_f1, az_f1, fpr, fnr, delta

print('Helper function defined.')

Helper function defined.


In [5]:
results = []

# 1. Full model baseline
print('Running: Full model (all 165 features)...')
r = run_xgb(X_train, y_train, X_val, y_val,
            X_azoo[feature_cols], y_azoo)
results.append({'Condition': 'Full model (165 features)', 'Type': 'Baseline',
                'N_features': 165, **dict(zip(['Krono_F1','AZ_F1','FPR','FNR','Delta_F1'], r))})
print(f'  KronoF1={r[0]:.4f}  AZ_F1={r[1]:.4f}  FPR={r[2]:.3f}  Δ={r[4]:.4f}')

# 2. Leave-one-group-out
for group_name, group_cols in GROUPS.items():
    remaining = [c for c in feature_cols if c not in group_cols]
    print(f'Running: Remove {group_name} ({len(remaining)} features remaining)...')
    r = run_xgb(X_train[remaining], y_train, X_val[remaining], y_val,
                X_azoo[remaining], y_azoo)
    results.append({'Condition': f'Remove {group_name}', 'Type': 'Leave-one-out',
                    'N_features': len(remaining),
                    **dict(zip(['Krono_F1','AZ_F1','FPR','FNR','Delta_F1'], r))})
    print(f'  KronoF1={r[0]:.4f}  AZ_F1={r[1]:.4f}  FPR={r[2]:.3f}  Δ={r[4]:.4f}')

# 3. Single-group-only
for group_name, group_cols in GROUPS.items():
    print(f'Running: Only {group_name} ({len(group_cols)} features)...')
    r = run_xgb(X_train[group_cols], y_train, X_val[group_cols], y_val,
                X_azoo[group_cols], y_azoo)
    results.append({'Condition': f'Only {group_name}', 'Type': 'Single-group',
                    'N_features': len(group_cols),
                    **dict(zip(['Krono_F1','AZ_F1','FPR','FNR','Delta_F1'], r))})
    print(f'  KronoF1={r[0]:.4f}  AZ_F1={r[1]:.4f}  FPR={r[2]:.3f}  Δ={r[4]:.4f}')

df_results = pd.DataFrame(results)
df_results.to_csv(RESULT_DIR / 'ablation_feature_groups.csv', index=False)
print('\nSaved to results/ablation/ablation_feature_groups.csv')

Running: Full model (all 165 features)...
  KronoF1=0.9535  AZ_F1=0.6955  FPR=0.422  Δ=-0.2580
Running: Remove Raw Counters (86 features remaining)...
  KronoF1=0.9479  AZ_F1=0.5761  FPR=0.675  Δ=-0.3719
Running: Remove Derived Features (129 features remaining)...
  KronoF1=0.9549  AZ_F1=0.6924  FPR=0.425  Δ=-0.2625
Running: Remove Temporal Features (143 features remaining)...
  KronoF1=0.9503  AZ_F1=0.7199  FPR=0.401  Δ=-0.2303
Running: Remove Sequence Features (137 features remaining)...
  KronoF1=0.9535  AZ_F1=0.6955  FPR=0.422  Δ=-0.2580
Running: Only Raw Counters (79 features)...
  KronoF1=0.9423  AZ_F1=0.7048  FPR=0.427  Δ=-0.2375
Running: Only Derived Features (36 features)...
  KronoF1=0.9353  AZ_F1=0.5966  FPR=0.635  Δ=-0.3387
Running: Only Temporal Features (22 features)...
  KronoF1=0.9139  AZ_F1=0.5075  FPR=0.783  Δ=-0.4064
Running: Only Sequence Features (28 features)...
  KronoF1=0.3341  AZ_F1=0.3333  FPR=0.000  Δ=-0.0007

Saved to results/ablation/ablation_feature_groups

In [6]:
# Pretty summary
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 40)
print('=== LEAVE-ONE-GROUP-OUT ===')
loo = df_results[df_results['Type'].isin(['Baseline','Leave-one-out'])][['Condition','N_features','Krono_F1','AZ_F1','FPR','FNR','Delta_F1']]
print(loo.to_string(index=False))
print()
print('=== SINGLE-GROUP-ONLY ===')
sg = df_results[df_results['Type']=='Single-group'][['Condition','N_features','Krono_F1','AZ_F1','FPR','FNR','Delta_F1']]
print(sg.to_string(index=False))

=== LEAVE-ONE-GROUP-OUT ===
                Condition  N_features  Krono_F1  AZ_F1    FPR    FNR  Delta_F1
Full model (165 features)         165    0.9535 0.6955 0.4217 0.1783   -0.2580
      Remove Raw Counters          86    0.9479 0.5761 0.6750 0.1037   -0.3719
  Remove Derived Features         129    0.9549 0.6924 0.4253 0.1807   -0.2625
 Remove Temporal Features         143    0.9503 0.7199 0.4013 0.1500   -0.2303
 Remove Sequence Features         137    0.9535 0.6955 0.4217 0.1783   -0.2580

=== SINGLE-GROUP-ONLY ===
             Condition  N_features  Krono_F1  AZ_F1    FPR    FNR  Delta_F1
     Only Raw Counters          79    0.9423 0.7048 0.4267 0.1527   -0.2375
 Only Derived Features          36    0.9353 0.5966 0.6353 0.1173   -0.3387
Only Temporal Features          22    0.9139 0.5075 0.7830 0.0803   -0.4064
Only Sequence Features          28    0.3341 0.3333 0.0000 1.0000   -0.0007
